# Y1 特征工程与排序预测流水线

## 最终结果

- 获胜单模型：`lightgbm_lambdarank`
- 官方验证 RankIC：`0.092940`
- 线性基线：`0.089678`
- TCN 基线：`0.091556`
- 采用特征阶段：`numeric`
- 采用缺失方案：`trend`

本 Notebook 仅预测 Y1，并从头到尾执行特征筛选、交叉项验证、类别数据消融、缺失方案对照、LambdaRank 调参、官方验证及预测。

## 背景与方法

### 关键假设

- 监督样本严格为 `mask_x & mask_y & finite(y1)`。
- Y1 不填补；测试期 `mask_y=True` 表示需要预测的评价股票池。
- 所有 lag、滚动统计、类别目标编码和缺失估计只使用当前时间以前的信息。
- 官方验证区间 `[2918, 3161)` 只用于最终晋级判断。
- 中间大数组使用磁盘 memmap；现有 Y1、Y2 Notebook 保持不变。

## 数据

### 1. 环境与可复现配置

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from y1_rank_pipeline_lib import PipelineConfig, run_pipeline

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)

config = PipelineConfig(
    data_path="../../data.z",
    output_dir="y1_rank_outputs",
    cache_dir=".cache_y1_rank",
    seed=42,
    tuning_trials=30,
    verify_top_parameter_sets=5,
    verbose=True,
)
config

PipelineConfig(data_path='data.z', output_dir='y1_rank_outputs', cache_dir='.cache_y1_rank', seed=42, top_feature_count=40, history_feature_count=20, interaction_pair_count=50, interaction_keep_count=30, correlation_time_samples=64, correlation_stock_cap=1200, discovery_train_time_samples=128, discovery_stock_cap=600, ablation_stock_cap=400, tuning_stock_cap=800, final_train_stock_cap=1200, tuning_trials=30, verify_top_parameter_sets=5, max_boost_rounds=3000, early_stopping_rounds=100, imputation_ic_threshold=0.0005, interaction_ic_threshold=0.001, cleanup_feature_matrices=True, verbose=True)

### 2. 执行完整流水线

此单元执行数据映射、特征发现、消融、调参、训练和预测。完整运行时间取决于 CPU、磁盘速度和 LightGBM 提前停止轮数。

In [2]:
result = run_pipeline(config)
result

[23:12:01] Loading competition data through read-only memmaps


[23:12:28] Discovering stable Y1 numeric features


D:\google_dl\book\友安杯\y1_rank_pipeline_lib.py:333: RuntimeWarning: Mean of empty slice
  means = np.nanmean(matrix, axis=0)
C:\Users\lenovo\.cache\codex-runtimes\codex-primary-runtime\dependencies\python\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


C:\Users\lenovo\.cache\codex-runtimes\codex-primary-runtime\dependencies\python\Lib\site-packages\numpy\lib\_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\lenovo\.cache\codex-runtimes\codex-primary-runtime\dependencies\python\Lib\site-packages\numpy\lib\_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


[23:12:42] Building reusable history, rank, and category-group caches


...

[06:35:46] final_test: 200/442 time points


[06:35:51] final_test: 300/442 time points


[06:35:58] final_test: 400/442 time points


[06:36:01] final_test: 442/442 time points


{'winner': 'lightgbm_lambdarank',
 'official_validation': {'time_points': 243,
  'mean_rank_ic': 0.09294016824452567,
  'std_rank_ic': 0.10905265458854116,
  'icir': 0.8522503977110106,
  'min_rank_ic': -0.22118338938430918,
  'max_rank_ic': 0.3441306786883424,
  'best_iteration': 8},
 'selected_stage': 'numeric',
 'selected_imputation': 'trend',
 'selected_numeric_features': [8,
  11,
  57,
  41,
  90,
  68,
  39,
  40,
  73,
  47,
  53,
  72,
  48,
  74,
  86,
  38,
  50,
  3,
  42,
  71,
  87,
  49,
  4,
  55,
  85,
  75,
  56,
  76,
  51,
  67,
  88,
  61,
  79,
  1,
  91,
  84,
  15,
  60,
  64,
  43],
 'selected_interaction_count': 0,
 'prediction': {'path': 'D:\\google_dl\\book\\友安杯\\y1_rank_outputs\\y1_best.npy',
  'shape': [442, 5282],
  'dtype': 'float32',
  'minimum': 0.0006291946046985686,
  'maximum': 1.0,
  'mean': 0.5,
  'neutral_count': 292324,
  'file_size_mb': 8.906082153320312,
  'np_load_verified': True},
 'runtime_seconds': 26668.320407390594,
 'output_dir': 'D:\\g

## 结果

### 3. 查看最终清单

In [3]:
manifest_path = Path("y1_rank_outputs/feature_manifest.json")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

summary = pd.Series(
    {
        "winner": manifest["winner"],
        "official_validation_rank_ic": manifest["official_validation"]["mean_rank_ic"],
        "linear_baseline_rank_ic": manifest["baselines"]["linear_mean_rank_ic"],
        "tcn_baseline_rank_ic": manifest["baselines"]["tcn_mean_rank_ic"],
        "selected_stage": manifest["selected_stage"],
        "selected_imputation": manifest["selected_imputation"],
        "selected_numeric_features": len(manifest["selected_numeric_features"]),
        "final_explicit_interactions": len(manifest["final_selected_interactions"]),
        "runtime_minutes": manifest["runtime_seconds"] / 60,
    },
    name="Y1 final result",
)
summary

winner                         lightgbm_lambdarank
official_validation_rank_ic                0.09294
linear_baseline_rank_ic                   0.089678
tcn_baseline_rank_ic                      0.091556
selected_stage                             numeric
selected_imputation                          trend
selected_numeric_features                       40
final_explicit_interactions                       0
runtime_minutes                         444.471433
Name: Y1 final result, dtype: object

### 4. 验证提交文件

In [4]:
prediction_path = Path("y1_rank_outputs/y1_best.npy")
predictions = np.load(prediction_path)

assert predictions.shape == (442, 5282)
assert predictions.dtype == np.float32
assert np.all(np.isfinite(predictions))

pd.Series(
    {
        "path": str(prediction_path.resolve()),
        "shape": str(predictions.shape),
        "dtype": str(predictions.dtype),
        "minimum": float(predictions.min()),
        "maximum": float(predictions.max()),
        "mean": float(predictions.mean()),
        "neutral_count": int(np.count_nonzero(predictions == 0.5)),
        "file_size_mb": prediction_path.stat().st_size / 1024**2,
    },
    name="submission validation",
)

path             D:\google_dl\book\友安杯\y1_rank_outputs\y1_best.npy
shape                                                  (442, 5282)
dtype                                                      float32
minimum                                                   0.000629
maximum                                                        1.0
mean                                                           0.5
neutral_count                                               292324
file_size_mb                                              8.906082
Name: submission validation, dtype: object

## 要点

- 以 `feature_manifest.json` 中的官方验证 RankIC 决定 LightGBM 是否晋级。
- `y1_best.npy` 是最终可提交文件，始终来自验证更优的单模型。
- `y1_experiment_report.md` 保存特征、交叉项、类别、缺失方案和模型比较的完整证据。
- 重新执行时，请使用 `jingge_ts` 内核从头运行本 Notebook；临时 memmap 缓存可删除并自动重建。